# Wick's theorem, verified by brute force

Executable companion to the Wick's theorem sections of chapter 3.

A vacuum expectation value of a string of fermion creation and annihilation
operators can be computed two entirely different ways:

1. **By anticommutation** — repeatedly apply
   $\{a_p, a_q^\dagger\}=\delta_{pq}$ to push annihilation operators to the
   right until they hit the vacuum. This is the elementary route of the
   worked examples in the text, and it branches badly.
2. **By Wick's theorem** — sum over all ways of pairing the operators into
   contractions, with a sign for each crossing. Only the *fully contracted*
   terms survive the vacuum expectation value.

This notebook implements both and checks that they agree. That check *is* the
content of the theorem.

Operators are written as `(index, dagger)` pairs, so `('p', True)` is
$a_p^\dagger$ and `('p', False)` is $a_p$. Indices stay symbolic, so results
come out as sums of products of Kronecker deltas.

In [ ]:
from itertools import combinations


class DeltaSum:
    """A linear combination of products of Kronecker deltas."""

    def __init__(self, terms=None):
        self.terms = dict(terms) if terms else {}

    @staticmethod
    def zero():
        return DeltaSum()

    @staticmethod
    def one():
        return DeltaSum({(): 1})

    @staticmethod
    def delta(p, q):
        if p == q:
            return DeltaSum.one()                 # delta_pp = 1
        return DeltaSum({((p, q) if p < q else (q, p),): 1})

    def __add__(self, other):
        out = dict(self.terms)
        for term, coeff in other.terms.items():
            out[term] = out.get(term, 0) + coeff
            if out[term] == 0:
                del out[term]
        return DeltaSum(out)

    def __mul__(self, other):
        if isinstance(other, int):
            return DeltaSum({t: c*other for t, c in self.terms.items()
                             if c*other != 0})
        out = {}
        for t1, c1 in self.terms.items():
            for t2, c2 in other.terms.items():
                term = tuple(sorted(set(t1) | set(t2)))
                out[term] = out.get(term, 0) + c1*c2
                if out[term] == 0:
                    del out[term]
        return DeltaSum(out)

    __rmul__ = __mul__

    def __eq__(self, other):
        return self.terms == other.terms

    def __bool__(self):
        return bool(self.terms)

    def __repr__(self):
        if not self.terms:
            return "0"
        pieces = []
        for term, coeff in sorted(self.terms.items()):
            body = "".join(f"d({p}{q})" for p, q in term) or "1"
            sign = "+" if coeff > 0 else "-"
            mag = "" if abs(coeff) == 1 else str(abs(coeff))
            pieces.append(f"{sign} {mag}{body}")
        text = " ".join(pieces)
        return text[2:] if text.startswith("+ ") else text

    def substitute(self, values):
        total = 0
        for term, coeff in self.terms.items():
            product = 1
            for p, q in term:
                product *= 1 if values.get(p, p) == values.get(q, q) else 0
                if product == 0:
                    break
            total += coeff*product
        return total


def c(index):
    """A creation operator a_index^dagger."""
    return (index, True)


def a(index):
    """An annihilation operator a_index."""
    return (index, False)

## Route 1: anticommutation

The recursion is the one carried out by hand in the text. Find the leftmost
place where an annihilation operator stands immediately to the left of a
creation operator and replace

$$
a_p a_q^\dagger \;\longrightarrow\; \delta_{pq} - a_q^\dagger a_p .
$$

When no such place remains the string is normal-ordered, and its vacuum
expectation value vanishes unless the string is empty.

In [ ]:
_call_count = [0]


def vev_bruteforce(ops):
    """<0| ops |0> by repeated use of the anticommutation relation."""
    ops = tuple(ops)
    _call_count[0] += 1

    if len(ops) == 0:
        return DeltaSum.one()
    if len(ops) % 2 == 1:            # an odd string can never pair up
        return DeltaSum.zero()
    if ops[0][1]:                    # starts with a creation operator
        return DeltaSum.zero()       #   -> kills the bra vacuum
    if not ops[-1][1]:               # ends with an annihilation operator
        return DeltaSum.zero()       #   -> kills the ket vacuum

    for k in range(len(ops) - 1):
        left, right = ops[k], ops[k+1]
        if not left[1] and right[1]:                  # a_p a_q^dagger
            rest = ops[:k] + ops[k+2:]
            swapped = ops[:k] + (right, left) + ops[k+2:]
            return (DeltaSum.delta(left[0], right[0]) * vev_bruteforce(rest)
                    + vev_bruteforce(swapped) * (-1))
    return DeltaSum.zero()           # normal-ordered and non-empty


def bruteforce_cost(ops):
    _call_count[0] = 0
    vev_bruteforce(ops)
    return _call_count[0]

## Route 2: Wick's theorem

Only one of the four elementary contractions survives,

$$
\overset{\frown}{a_p a_q^\dagger} = \delta_{pq},
\qquad
\overset{\frown}{a_p a_q} =
\overset{\frown}{a_q^\dagger a_p} =
\overset{\frown}{a_p^\dagger a_q^\dagger} = 0 ,
$$

and each set of contractions carries the sign $(-1)^{\nu}$ with $\nu$ the
number of crossings of the contraction lines.

In [ ]:
def contraction(x, y):
    """<0| x y |0>: non-zero only for an annihilation operator to the left
    of a creation operator."""
    if (not x[1]) and y[1]:
        return DeltaSum.delta(x[0], y[0])
    return DeltaSum.zero()


def perfect_matchings(indices):
    """All ways of pairing up a list of positions."""
    if not indices:
        yield []
        return
    first, rest = indices[0], indices[1:]
    for k in range(len(rest)):
        remainder = rest[:k] + rest[k+1:]
        for tail in perfect_matchings(remainder):
            yield [(first, rest[k])] + tail


def matching_sign(pairs):
    """(-1) to the number of crossings of the contraction lines."""
    crossings = 0
    for (p, q), (r, s) in combinations(pairs, 2):
        if p < r < q < s or r < p < s < q:
            crossings += 1
    return (-1)**crossings


def vev_wick(ops):
    """<0| ops |0> as a signed sum over fully contracted terms."""
    ops = tuple(ops)
    if len(ops) % 2 == 1:
        return DeltaSum.zero()
    total = DeltaSum.zero()
    for pairs in perfect_matchings(list(range(len(ops)))):
        value = DeltaSum.one()
        for i, j in pairs:
            value = value * contraction(ops[i], ops[j])
            if not value:
                break
        if value:
            total = total + value * matching_sign(pairs)
    return total


def double_factorial(m):
    """(m-1)!! -- the number of perfect matchings of m objects."""
    out, k = 1, m - 1
    while k > 1:
        out *= k
        k -= 2
    return out

## Two operators

The whole content of the theorem at $M=2$ is
$xy = N[xy] + \overset{\frown}{xy}$, and only one of the four combinations
has a non-vanishing contraction.

In [ ]:
cases = (((a("p"), c("q")), "<0| a_p a_q^+ |0>"),
         ((a("p"), a("q")), "<0| a_p a_q   |0>"),
         ((c("p"), a("q")), "<0| a_p^+ a_q |0>"),
         ((c("p"), c("q")), "<0| a_p^+ a_q^+ |0>"))
for ops, label in cases:
    brute, wick = vev_bruteforce(ops), vev_wick(ops)
    print(f"{label:22s} = {str(brute):10s} (Wick: {str(wick):10s} "
          f"agree: {brute == wick})")

## Three operators

An odd number of operators can never be paired up completely, so at least one
is always left uncontracted — and $\langle 0|a|0\rangle=\langle
0|a^\dagger|0\rangle=0$.

In [ ]:
for ops, label in (((a("p"), c("q"), c("r")), "<0| a_p a_q^+ a_r^+ |0>"),
                   ((a("p"), a("q"), c("r")), "<0| a_p a_q a_r^+   |0>"),
                   ((c("p"), a("q"), c("r")), "<0| a_p^+ a_q a_r^+ |0>")):
    brute, wick = vev_bruteforce(ops), vev_wick(ops)
    print(f"{label:24s} = {brute}   (Wick: {wick}, agree: {brute == wick})")

## Four operators: the overlap $\langle rs|pq\rangle$

Of the three ways to pair four operators, one vanishes on inspection, one is
nested (no crossings, plus sign) and one crosses once (minus sign). The result
is the antisymmetrised overlap

$$
\langle rs|pq\rangle = \delta_{rp}\delta_{sq}-\delta_{sp}\delta_{rq}.
$$

In [ ]:
ops = (a("s"), a("r"), c("p"), c("q"))
brute, wick = vev_bruteforce(ops), vev_wick(ops)
print("<0| a_s a_r a_p^+ a_q^+ |0>")
print(f"  by anticommutation : {brute}")
print(f"  by Wick's theorem  : {wick}")
print(f"  agree              : {brute == wick}")
print()
print("the three pairings, one at a time:")
labels = {((0, 1), (2, 3)): "(a_s,a_r)(a_p^+,a_q^+)  both contractions vanish",
          ((0, 3), (1, 2)): "(a_s,a_q^+)(a_r,a_p^+)  nested, no crossing",
          ((0, 2), (1, 3)): "(a_s,a_p^+)(a_r,a_q^+)  one crossing"}
for pairs in perfect_matchings([0, 1, 2, 3]):
    value = DeltaSum.one()
    for i, j in pairs:
        value = value * contraction(ops[i], ops[j])
    key = tuple(sorted(tuple(sorted(p)) for p in pairs))
    signed = value * matching_sign(pairs)
    print(f"  {labels.get(key, str(key)):40s} -> {signed}")

## Six and eight operators

The agreement is not an accident of small numbers.

In [ ]:
alphabet = "abcdefghijklmnop"
for m in (2, 4, 6, 8):
    half = m // 2
    names = alphabet[:m]
    ops = tuple(a(n) for n in names[:half]) + tuple(c(n) for n in names[half:])
    brute, wick = vev_bruteforce(ops), vev_wick(ops)
    print(f"M = {m}: {len(brute.terms):3d} surviving terms, "
          f"(M-1)!! = {double_factorial(m):3d} pairings, "
          f"agree: {brute == wick}")

## The number operator

With $|12\rangle = a_1^\dagger a_2^\dagger|0\rangle$ and
$\hat N = \sum_i a_i^\dagger a_i$, Wick's theorem should return the particle
number.

In [ ]:
total = 0
for i in ("1", "2", "3", "4"):
    ops = (a("2"), a("1"), c(i), a(i), c("1"), c("2"))
    value = vev_wick(ops).substitute({})
    total += value
    print(f"  i = {i}: <12| a_{i}^+ a_{i} |12> = {value}")
print(f"\n  sum over i = {total}   (= N, the particle number)")

## The two-body interaction

$$
\hat H_I=\tfrac12\sum_{pqrs}\langle pq|v|rs\rangle\,
a_p^\dagger a_q^\dagger a_s a_r,
\qquad
|ij\rangle = a_i^\dagger a_j^\dagger|0\rangle .
$$

We evaluate the eight-operator string for every assignment of $(p,q,r,s)$ and
see which survive. The answer should be the antisymmetrised matrix element
$\langle ij|v|ij\rangle - \langle ij|v|ji\rangle$.

In [ ]:
surviving = {}
for p in ("i", "j"):
    for q in ("i", "j"):
        for r in ("i", "j"):
            for s in ("i", "j"):
                ops = (a("j"), a("i"), c(p), c(q), a(s), a(r), c("i"), c("j"))
                value = vev_wick(ops).substitute({})
                if value:
                    surviving[(p, q, r, s)] = value

for (p, q, r, s), value in sorted(surviving.items()):
    print(f"  <{p}{q}|v|{r}{s}>  coefficient {value:+d}")

direct = sum(v for k, v in surviving.items()
             if k in (("i", "j", "i", "j"), ("j", "i", "j", "i")))
exchange = sum(v for k, v in surviving.items()
               if k in (("i", "j", "j", "i"), ("j", "i", "i", "j")))
print()
print(f"  with the factor 1/2:  {direct//2:+d} <ij|v|ij>  "
      f"{exchange//2:+d} <ij|v|ji>")
print("  so <ij|H_I|ij> = <ij|v|ij> - <ij|v|ji> = <ij|v|ij>_AS")

## Why the theorem is worth having

Both routes grow quickly, but they grow differently. The anticommutation
recursion branches with no structure one can anticipate. Wick's theorem
enumerates a known number of terms in advance, most of which vanish on
inspection because one of their contractions is zero, and the survivors can be
written down directly.

In [ ]:
print(f"{'M':>4s} {'pairings (M-1)!!':>18s} {'recursive calls':>17s} "
      f"{'surviving terms':>17s}")
for m in (2, 4, 6, 8, 10):
    half = m // 2
    names = alphabet[:m]
    ops = tuple(a(n) for n in names[:half]) + tuple(c(n) for n in names[half:])
    print(f"{m:4d} {double_factorial(m):18d} {bruteforce_cost(ops):17d} "
          f"{len(vev_wick(ops).terms):17d}")

## Wick's generalised theorem

In practice we never meet a bare string of operators.  We meet a product of
*groups*, each of them already normal-ordered: the bra, the operator and the
ket.  The generalised theorem says that for such a product

$$
N[A_1A_2\cdots]\,N[B_1B_2\cdots]\,N[C_1C_2\cdots]
 = N[A_1A_2\cdots B_1B_2\cdots C_1C_2\cdots]
 + \sum_{\rm all} N[A_1A_2\cdots C_1C_2\cdots],
$$

where the sum runs over all contractions **between different groups**.

The contractions *inside* a group are absent for a reason, and the reason is
worth stating separately because it is the whole proof in miniature.  A group
in normal-ordered form has all its creation operators on the left.  Pick two
of its operators, $A_k$ and $A_l$ with $k<l$.  If $A_k$ is a creation
operator then $A_l$ is either another creation operator or an annihilation
operator standing to its right, and both contractions vanish.  If $A_k$ is an
annihilation operator then so is everything to its right, and that contraction
vanishes too.  Internal contractions are not thrown away — they are zero.

In [ ]:
def normal_order(ops):
    """Bring a string into normal-ordered form: (sign, reordered)."""
    ops = tuple(ops)
    creators = [o for o in ops if o[1]]
    annihilators = [o for o in ops if not o[1]]
    swaps = sum(1 for i, o in enumerate(ops) if not o[1]
                for p in ops[i + 1:] if p[1])
    return (-1) ** swaps, tuple(creators + annihilators)


def is_normal_ordered(ops):
    """True if no annihilation operator stands to the left of a creator."""
    seen = False
    for _, dagger in ops:
        if dagger and seen:
            return False
        if not dagger:
            seen = True
    return True


def internal_contractions(group):
    """The contractions of a group with itself -- all of them should vanish."""
    return [contraction(group[i], group[j])
            for i in range(len(group)) for j in range(i + 1, len(group))]


def _inversions(sequence):
    return sum(1 for i, x in enumerate(sequence)
               for y in sequence[i + 1:] if y < x)


def contraction_sign(pairs, n_ops):
    """Sign of a (possibly partial) set of contractions.

    Bring the contracted operators together, leaving the uncontracted ones in
    their original relative order, and count the interchanges.  For a fully
    contracted term this is the (-1)^(crossings) of matching_sign().
    """
    used = {i for pair in pairs for i in pair}
    order = [i for pair in pairs for i in pair]
    order += [k for k in range(n_ops) if k not in used]
    return (-1) ** _inversions(order)


def contraction_sets(n_ops, allowed=None):
    """All sets of disjoint contractions, from the empty set upwards."""
    def build(decided):
        free = [k for k in range(n_ops) if k not in decided]
        if not free:
            yield []
            return
        first, rest = free[0], free[1:]
        for tail in build(decided | {first}):        # first left uncontracted
            yield tail
        for partner in rest:                         # or contracted
            if allowed and not allowed(first, partner):
                continue
            for tail in build(decided | {first, partner}):
                yield [(first, partner)] + tail
    return build(frozenset())


def flatten(groups):
    """Concatenate the groups, recording which group each operator came from."""
    ops, labels = [], []
    for number, group in enumerate(groups):
        for operator in group:
            ops.append(operator)
            labels.append(number)
    return tuple(ops), labels


def vev_generalised(groups):
    """<0| N[A...] N[B...] ... |0> from the generalised theorem."""
    ops, labels = flatten(groups)
    n_ops = len(ops)
    total = DeltaSum.zero()
    for pairs in contraction_sets(n_ops, lambda i, j: labels[i] != labels[j]):
        if 2 * len(pairs) != n_ops:            # not fully contracted
            continue
        value = DeltaSum.one()
        for i, j in pairs:
            value = value * contraction(ops[i], ops[j])
            if not value:
                break
        if value:
            total = total + value * contraction_sign(pairs, n_ops)
    return total


def count_matchings(sizes):
    """(all perfect matchings, those with no line inside a group)."""
    labels = [n for n, size in enumerate(sizes) for _ in range(size)]
    n_ops = len(labels)
    every = inter = 0
    for pairs in perfect_matchings(list(range(n_ops))):
        every += 1
        if all(labels[i] != labels[j] for i, j in pairs):
            inter += 1
    return every, inter

### The groups of the one- and two-body matrix elements

For $\langle ij|\hat O^{(1)}|k\ell\rangle$ the groups are the bra
$a_ja_i$, the operator $a_p^\dagger a_q$ and the ket
$a_k^\dagger a_\ell^\dagger$; for the two-body element the operator has four
operators instead of two.  All of them are normal-ordered as written, so all
their internal contractions vanish, and the number of surviving pairings drops
sharply.

In [ ]:
one_body = [[a("j"), a("i")], [c("p"), a("q")], [c("k"), c("l")]]
two_body = [[a("j"), a("i")], [c("p"), c("q"), a("s"), a("r")],
            [c("k"), c("l")]]

for name, groups in (("one-body", one_body), ("two-body", two_body)):
    ordered = all(is_normal_ordered(g) for g in groups)
    internal = [x for g in groups for x in internal_contractions(g) if x]
    print(f"{name}: groups normal-ordered: {ordered}, "
          f"non-vanishing internal contractions: {len(internal)}")

print()
print(f"{'groups':>10s} {'M':>4s} {'(M-1)!! pairings':>18s} {'between groups':>16s}")
for name, groups in (("one-body", one_body), ("two-body", two_body)):
    sizes = [len(g) for g in groups]
    every, inter = count_matchings(sizes)
    print(f"{'+'.join(map(str, sizes)):>10s} {sum(sizes):4d} {every:18d} {inter:16d}")

### Three routes to the same matrix element

Brute-force anticommutation, the ordinary theorem and the generalised theorem
must all give the same answer.  The generalised theorem gets there with a
fraction of the terms.

In [ ]:
for name, groups in (("one-body", one_body), ("two-body", two_body)):
    ops, _ = flatten(groups)
    brute = vev_bruteforce(ops)
    wick = vev_wick(ops)
    general = vev_generalised(groups)
    print(f"{name}:  anticommutation == Wick: {brute == wick},  "
          f"Wick == generalised: {wick == general}")
    print(f"   {general}")
    print()

### The theorem as an identity between operators

Everything so far lives inside a vacuum expectation value.  The generalised
theorem is stronger than that: it is an identity between *operators*, and the
uncontracted and partially contracted terms are as much a part of it as the
fully contracted ones.  To check it in that stronger form we represent the
creation and annihilation operators as matrices on the $2^n$ states of a small
Fock space, with the Jordan-Wigner sign
$(-1)^{\text{occupied orbitals below }p}$ keeping the anticommutation
relations exact, and compare the two sides entry by entry.

In [ ]:
import numpy as np


class FockSpace:
    """Fermion operators as matrices on the 2^n states of a Fock space."""

    def __init__(self, n_orbitals):
        self.n_orbitals = n_orbitals
        self.dim = 1 << n_orbitals

    def annihilate(self, p):
        matrix = np.zeros((self.dim, self.dim))
        for state in range(self.dim):
            if state & (1 << p):
                sign = (-1) ** bin(state & ((1 << p) - 1)).count("1")
                matrix[state ^ (1 << p), state] = sign
        return matrix

    def create(self, p):
        return self.annihilate(p).T

    def operator(self, op):
        index, dagger = op
        return self.create(index) if dagger else self.annihilate(index)

    def string(self, ops):
        matrix = np.eye(self.dim)
        for op in ops:
            matrix = matrix @ self.operator(op)
        return matrix

    def normal_product(self, ops):
        """N[ops] as a matrix: reorder, and carry the sign along."""
        sign, reordered = normal_order(ops)
        return sign * self.string(reordered)

    def check_generalised(self, groups):
        """max |LHS - RHS| of the generalised theorem."""
        lhs = np.eye(self.dim)
        for group in groups:
            lhs = lhs @ self.normal_product(group)

        ops, labels = flatten(groups)
        n_ops = len(ops)
        rhs = np.zeros((self.dim, self.dim))
        for pairs in contraction_sets(n_ops,
                                      lambda i, j: labels[i] != labels[j]):
            value = 1
            for i, j in pairs:
                value *= contraction(ops[i], ops[j]).substitute({})
                if value == 0:
                    break
            if value == 0:
                continue
            used = {i for pair in pairs for i in pair}
            rest = [ops[k] for k in range(n_ops) if k not in used]
            remainder = (np.eye(self.dim) if not rest
                         else self.normal_product(rest))
            rhs = rhs + contraction_sign(pairs, n_ops) * value * remainder
        return float(np.abs(lhs - rhs).max())

In [ ]:
space = FockSpace(4)

cases = [
    ("N[a_1 a_0] N[a_0^+ a_2] N[a_2^+ a_3^+]",
     [[a(1), a(0)], [c(0), a(2)], [c(2), c(3)]]),
    ("N[a_1 a_0] N[a_0^+ a_1^+ a_3 a_2] N[a_2^+ a_3^+]",
     [[a(1), a(0)], [c(0), c(1), a(3), a(2)], [c(2), c(3)]]),
    ("N[a_0^+ a_1] N[a_1^+ a_0]",
     [[c(0), a(1)], [c(1), a(0)]]),
    ("N[a_0^+ a_1^+ a_1 a_0] N[a_2^+ a_3] N[a_3^+ a_2]",
     [[c(0), c(1), a(1), a(0)], [c(2), a(3)], [c(3), a(2)]]),
]
for label, groups in cases:
    print(f"max |LHS - RHS| = {space.check_generalised(groups):.1e}   {label}")

# ordinary Wick is the special case of one operator per group
single = [a(0), c(1), a(2), c(0), c(2), a(1)]
residual = space.check_generalised([[op] for op in single])
print(f"max |LHS - RHS| = {residual:.1e}   ordinary Wick, "
      f"{len(single)} operators")

## Where this leads

Nothing in the argument used the fact that $|0\rangle$ is the true vacuum.  In
many-body theory the reference state is a filled Fermi sea $|c\rangle$, and the
useful statement is Wick's theorem relative to *that* reference, obtained by
the same argument once the creation and annihilation operators are redefined
with respect to the new vacuum, as in the particle-hole formalism.  The only
change is in the elementary contractions: with respect to $|c\rangle$ both
$\langle c|a_a a_b^\dagger|c\rangle=\delta_{ab}$ for particle states above the
Fermi level and $\langle c|a_i^\dagger a_j|c\rangle=\delta_{ij}$ for hole
states below it are non-zero.

That particle-hole form of the generalised theorem is what generates the
Hartree-Fock equations, the perturbation expansion and the coupled-cluster
amplitude equations of the chapters ahead.  The counting above is the reason
those derivations are possible at all: of the pairings of eight operators,
only a quarter join different groups, and of those only a handful survive the
elementary contractions.